In [1]:
# imports
import requests
import pandas as pd
from io import StringIO
from sqlalchemy import create_engine
import psycopg2
from dotenv import load_dotenv
import os
import datetime
from utils import *

## API request

In [2]:
endpoint_url = 'https://data.cityofnewyork.us/resource/h9gi-nx95.csv'

params = {
    "$limit" : 1000
}

r = requests.get(endpoint_url, params=params)
print(r.status_code)

200


In [3]:
print(r)
print(r.headers)
print(r.status_code)

<Response [200]>
{'Server': 'nginx', 'Date': 'Sun, 17 May 2026 17:33:59 GMT', 'Content-Type': 'text/csv; charset=UTF-8', 'Transfer-Encoding': 'chunked', 'Connection': 'keep-alive', 'Access-Control-Allow-Origin': '*', 'ETag': '"Zm94dHJvdC4xNTY3NV80MzNfODA4Njk2eEw1ZWYzaWlUQlhyczl2bFlEeWpqcWw2NHc---gzirqip3BbWj5NgELI2UY8lBOmYAQuA--gzip--gzip"', 'X-SODA2-Fields': '["crash_date","crash_time","borough","zip_code","latitude","longitude","location","on_street_name","off_street_name","cross_street_name","number_of_persons_injured","number_of_persons_killed","number_of_pedestrians_injured","number_of_pedestrians_killed","number_of_cyclist_injured","number_of_cyclist_killed","number_of_motorist_injured","number_of_motorist_killed","contributing_factor_vehicle_1","contributing_factor_vehicle_2","contributing_factor_vehicle_3","contributing_factor_vehicle_4","contributing_factor_vehicle_5","collision_id","vehicle_type_code1","vehicle_type_code2","vehicle_type_code_3","vehicle_type_code_4","vehicle_

In [4]:
if r.status_code == 200:
    data = StringIO(r.text)
    df = pd.read_csv(data)

In [5]:
df

,crash_date,crash_time,borough,zip_code,latitude,longitude,location,on_street_name,off_street_name,cross_street_name,...,contributing_factor_vehicle_2,contributing_factor_vehicle_3,contributing_factor_vehicle_4,contributing_factor_vehicle_5,collision_id,vehicle_type_code1,vehicle_type_code2,vehicle_type_code_3,vehicle_type_code_4,vehicle_type_code_5
0,2021-09-11T00:00:00.000,2:39,NaN,NaN,NaN,NaN,NaN,WHITESTONE EXPRESSWAY,20 AVENUE,NaN,...,Unspecified,NaN,NaN,NaN,4455765,Sedan,Sedan,NaN,NaN,NaN
1,2022-03-26T00:00:00.000,11:45,NaN,NaN,NaN,NaN,NaN,QUEENSBORO BRIDGE UPPER,NaN,NaN,...,NaN,NaN,NaN,NaN,4513547,Sedan,NaN,NaN,NaN,NaN
2,2023-11-01T00:00:00.000,1:29,BROOKLYN,11230.0,40.621790,-73.970024,"\n, \n(40.62179, -73.970024)",OCEAN PARKWAY,AVENUE K,NaN,...,Unspecified,Unspecified,NaN,NaN,4675373,Moped,Sedan,Sedan,NaN,NaN
3,2022-06-29T00:00:00.000,6:55,NaN,NaN,NaN,NaN,NaN,THROGS NECK BRIDGE,NaN,NaN,...,Unspecified,NaN,NaN,NaN,4541903,Sedan,Pick-up Truck,NaN,NaN,NaN
4,2022-09-21T00:00:00.000,13:21,NaN,NaN,NaN,NaN,NaN,BROOKLYN BRIDGE,NaN,NaN,...,Unspecified,NaN,NaN,NaN,4566131,Station Wagon/Sport Utility Vehicle,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,2021-04-14T00:00:00.000,19:01,BROOKLYN,11226.0,40.648840,-73.951020,"\n, \n(40.64884, -73.95102)",NaN,NaN,2820 SNYDER AVENUE,...,Unspecified,NaN,NaN,NaN,4407358,Station Wagon/Sport Utility Vehicle,Station Wagon/Sport Utility Vehicle,NaN,NaN,NaN
996,2021-09-11T00:00:00.000,13:30,QUEENS,11361.0,40.757725,-73.779274,"\n, \n(40.757725, -73.779274)",NORTHERN BOULEVARD,204 STREET,NaN,...,Unspecified,NaN,NaN,NaN,4455876,Sedan,Sedan,NaN,NaN,NaN
997,2021-04-15T00:00:00.000,10:13,MANHATTAN,10021.0,40.766586,-73.953100,"\n, \n(40.766586, -73.9531)",NaN,NaN,525 EAST 72 STREET,...,Unspecified,NaN,NaN,NaN,4407695,3-Door,Station Wagon/Sport Utility Vehicle,NaN,NaN,NaN
998,2021-04-14T00:00:00.000,9:40,BRONX,10465.0,40.833126,-73.828285,"\n, \n(40.833126, -73.828285)",BRUCKNER EXPRESSWAY,EAST TREMONT AVENUE,NaN,...,NaN,NaN,NaN,NaN,4407265,Station Wagon/Sport Utility Vehicle,NaN,NaN,NaN,NaN


## Connect to Postgres database

In [6]:
load_dotenv()
host = os.getenv("DB_host")
user = os.getenv("DB_user")
password = os.getenv("DB_password")
db = os.getenv("DB_name")
port = os.getenv("DB_port")

In [7]:
engine = create_engine(f"postgresql+psycopg2://{user}:{password}@{host}/{db}")

# for raw sql queries
conn_string = f"host={host} dbname={db} user={user} password={password}"
conn = psycopg2.connect(conn_string)


In [8]:
type(engine)

sqlalchemy.engine.base.Engine

## Ingest dataframe to database

Two options of inserting to db:

In [9]:
# 1. for simple insert operations only, not for combining with other sql queries
df.to_sql(name='crashes', con=engine, if_exists='append', index=False)

1000

In [ ]:
# 2. using sqlalchemy connection object with commit() method to control is executed and what crashes
with engine.connect() as conn:
    df.to_sql(name='crashes', con=conn, if_exists='append', index=False)
    conn.execute("UPDATE ...")
    conn.execute("INSERT ...")
    conn.commit()  # all 3 succeed together, or none do

## Pulling data only from after the last download date

In [11]:
r.headers['Date']

'Sun, 17 May 2026 17:33:59 GMT'

In [12]:
download_date = pd.to_datetime(r.headers['Date'])
df['download_timestamp'] = download_date

In [13]:
df['crash_date'][0]

'2021-09-11T00:00:00.000'

In [14]:
# d = datetime.timedelta(days=10)
# temp_date = download_date - d
# print(temp_date)


last_api_call = datetime.datetime.isoformat(download_date, timespec='milliseconds')
print(type(last_api_call))
last_api_call

<class 'str'>


'2026-05-17T17:33:59.000'

In [15]:
endpoint_url = 'https://data.cityofnewyork.us/resource/h9gi-nx95.csv'

params = {
    "$limit" : 1000,
    "$where" : f"crash_date > '{last_api_call}'"
}

r = requests.get(endpoint_url, params=params)
print(r.status_code)

200


In [16]:
print(type(r.status_code))

<class 'int'>


In [17]:
type(r)

requests.models.Response

In [18]:
# workflow to test functions in utils.py

endpoint_url = 'https://data.cityofnewyork.us/resource/h9gi-nx95.csv'
last_api_call = datetime.datetime.isoformat(download_date, timespec='milliseconds')
rows = 1000

r = request_crash_data(endpoint_url, rows, last_api_call)

if r.status_code == 200:
    print('The API request was succesful.')
    df = create_df_to_load(r)

else:
    print(f'The API request failed, status_code: {r.status_code}.')



The API request was succesful.
